# 🔬 Estudo de Ablação Quadruplo: Comparação Estrutural de Performance (XGBoost)

**Projeto de TCC:** Detecção de Fraude em Tempo Real com Modelos Tabulares Aumentados por Grafos  
**Dataset:** IEEE-CIS Fraud Detection Benchmark (590.540 transações)  
**Objetivo do Experimento:** Avaliar e isolar as contribuições individuais e combinadas de cada família de atributos usando o classificador **XGBoost** sob 4 cenários rigorosamente controlados:
1. **Configuração A: XGBoost Baseline**: Apenas variáveis tabulares originais (sem normalizações temporais, sem grafos e sem agregações).
2. **Configuração B: XGBoost + ONLY Graph Features**: Variáveis originais adicionadas *apenas* de atributos de rede/conectividade de grafos (PageRank, Graus e Velocidade 24h).
3. **Configuração C: XGBoost + ONLY Feature Engineering**: Variáveis originais adicionadas *apenas* de engenharia clássica/Magic Features (D normalizados, UIDs e desvios de ticket por cliente).
4. **Configuração D: All-in-One XGBoost**: Todos os atributos combinados (Originais + Grafos + Magic Features).


---
## 1. Instalação de Pacotes & Configuração do Ambiente


In [ ]:
# Instalação dos pacotes necessários no ambiente do Google Colab
!pip install -q xgboost lightgbm networkx scikit-learn torch torch_geometric matplotlib seaborn


In [ ]:
import os
import gc
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, roc_curve

import xgboost as xgb

try:
    from IPython.display import display
except ImportError:
    display = print

# Configuração de Plots
plt.rcParams.update({'figure.dpi': 110, 'font.family': 'DejaVu Sans'})
sns.set_style('whitegrid')

PALETTE = {
    'baseline': '#9E9E9E',
    'only_graph': '#E63946',
    'only_fe': '#457B9D',
    'all_in_one': '#1D3557'
}

print("✅ Ambiente configurado com sucesso para o Estudo de Ablação.")


---
## 2. Ingestão de Dados & Otimização de Memória RAM (Downcasting)


In [ ]:
# Se estiver usando o Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/ieee-data'

# Se fez upload direto no Colab:
DATA_DIR = '.'

train_trans_path = os.path.join(DATA_DIR, 'train_transaction.csv')
train_id_path = os.path.join(DATA_DIR, 'train_identity.csv')

def reduce_mem_usage(df, verbose=True):
    """Converte tipos numéricos para float32/int32 para economizar RAM."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and not pd.api.types.is_categorical_dtype(df[col]):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"RAM: {start_mem:.2f} MB reduzido para {end_mem:.2f} MB ({100*(start_mem-end_mem)/start_mem:.1f}% economia)")
    return df

print("Carregando 100% dos dados do IEEE-CIS...")
df_trans = pd.read_csv(train_trans_path)
df_trans = reduce_mem_usage(df_trans)

df_id = pd.read_csv(train_id_path)
df_id = reduce_mem_usage(df_id)

df = df_trans.merge(df_id, on='TransactionID', how='left')
del df_trans, df_id
gc.collect()

df = df.rename(columns={"TransactionDT": "timestamp"})
print(f"Dataset carregado: {df.shape[0]:,} transações × {df.shape[1]} colunas.")

# Split Temporal Causal (80% treino / 20% validação)
split_idx = int(len(df) * 0.8)
df = df.sort_values('timestamp').reset_index(drop=True)

train_df_raw = df.iloc[:split_idx].copy()
val_df_raw = df.iloc[split_idx:].copy()
del df
gc.collect()

print(f"Treino   : {len(train_df_raw):,} linhas (Fraude: {train_df_raw['isFraud'].mean():.3%})")
print(f"Validação: {len(val_df_raw):,} linhas (Fraude: {val_df_raw['isFraud'].mean():.3%})")


---
## 3. Engenharia de Features Avançadas e PCA


In [ ]:
print("Construindo Atributos Tabulares Otimizados...")

train_df = train_df_raw.copy()
val_df = val_df_raw.copy()

# Normalização de D
for d_col in ['D1', 'D2', 'D10', 'D15']:
    if d_col in train_df.columns:
        train_df[f'{d_col}n'] = (train_df['timestamp'] / 86400.0) - train_df[d_col]
        val_df[f'{d_col}n'] = (val_df['timestamp'] / 86400.0) - val_df[d_col]

# UID Tabular
def build_uid(df):
    c1 = df['card1'].fillna(0).astype(int).astype(str)
    addr = df['addr1'].fillna(0).astype(int).astype(str)
    day = df['timestamp'] / 86400.0
    d1_anchor = (day - df['D1'].fillna(0)).astype(int).astype(str)
    return c1 + "_" + addr + "_" + d1_anchor

train_df['uid'] = build_uid(train_df)
val_df['uid'] = build_uid(val_df)

# Agregações de Grupo Tabulares (Ajustadas estritamente no Treino)
agg_cols = ['TransactionAmt', 'D15n', 'C1', 'C13']
for col in agg_cols:
    if col in train_df.columns:
        mean_map = train_df.groupby('uid')[col].mean().to_dict()
        std_map = train_df.groupby('uid')[col].std().fillna(0.0).to_dict()
        global_mean = train_df[col].mean()
        
        for d in [train_df, val_df]:
            d_mean = d['uid'].map(mean_map).fillna(global_mean)
            d_std = d['uid'].map(std_map).fillna(0.0)
            d[f'{col}_to_mean_uid'] = d[col] / (d_mean + 1e-5)
            d[f'{col}_to_std_uid'] = (d[col] - d_mean) / (d_std + 1e-5)

# Compressão PCA das 339 variáveis V em 15 componentes
v_cols = [c for c in train_df.columns if c.startswith('V') and c[1:].isdigit()]
medians_v = train_df[v_cols].median().fillna(0.0)
scaler = StandardScaler()
X_v_train = scaler.fit_transform(train_df[v_cols].fillna(medians_v))
X_v_val = scaler.transform(val_df[v_cols].fillna(medians_v))

pca = PCA(n_components=15, random_state=42)
pca_train = pca.fit_transform(X_v_train)
pca_val = pca.transform(X_v_val)

for i in range(15):
    train_df[f'PCA_V_{i}'] = pca_train[:, i]
    val_df[f'PCA_V_{i}'] = pca_val[:, i]

train_df = train_df.drop(columns=v_cols)
val_df = val_df.drop(columns=v_cols)
del X_v_train, X_v_val, pca_train, pca_val
gc.collect()

# Pré-processamento: Eliminação de colunas >90% nulas e Label Encoding
null_rates = train_df.isnull().mean()
high_null_cols = [c for c in null_rates[null_rates > 0.90].index if c not in ['isFraud', 'TransactionID']]
train_df = train_df.drop(columns=high_null_cols)
val_df = val_df.drop(columns=high_null_cols)

# Guardar se tinha DeviceType antes do encoding para o roteamento de identidade
train_has_dev = train_df['DeviceType'].notnull()
val_has_dev = val_df['DeviceType'].notnull()

cat_cols = [c for c in train_df.select_dtypes(include=['object', 'category']).columns if c not in ['uid', 'TransactionID']]
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    val_df[col] = val_df[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

print("✅ Base tabular e encoders configurados.")


---
## 4. Engenharia de Features de Grafo


In [ ]:
print("Construindo Features de Rede...")

# 1. Grafo Bipartido UID ↔ Dispositivo
train_df['DeviceInfo_clean'] = train_df_raw['DeviceInfo'].fillna('null_device').astype(str)
val_df['DeviceInfo_clean'] = val_df_raw['DeviceInfo'].fillna('null_device').astype(str)

G = nx.Graph()
for _, row in train_df[train_df['DeviceInfo_clean'] != 'null_device'].iterrows():
    G.add_edge(f"uid_{row['uid']}", f"dev_{row['DeviceInfo_clean']}")

try:
    pr = nx.pagerank(G, alpha=0.85)
except Exception:
    pr = {}

default_pr = 1.0 / max(len(G), 1)
train_df['uid_pagerank'] = train_df['uid'].apply(lambda x: pr.get(f"uid_{x}", default_pr))
val_df['uid_pagerank'] = val_df['uid'].apply(lambda x: pr.get(f"uid_{x}", default_pr))

# Graus de conectividade de rede
uid_degrees = train_df['uid'].value_counts().to_dict()
dev_degrees = train_df['DeviceInfo_clean'].value_counts().to_dict()

train_df['uid_degree'] = train_df['uid'].map(uid_degrees).fillna(0)
val_df['uid_degree'] = val_df['uid'].map(uid_degrees).fillna(0)

train_df['device_degree'] = train_df['DeviceInfo_clean'].map(dev_degrees).fillna(0)
val_df['device_degree'] = val_df['DeviceInfo_clean'].map(dev_degrees).fillna(0)

# 2. Velocidade de Rede 24h
def add_velocity(df, col_name):
    df_sorted = df.sort_values('timestamp')
    times = df_sorted['timestamp'].values
    left_idx = np.searchsorted(times, times - 86400, side='left')
    df[col_name] = np.arange(len(times)) - left_idx + 1
    return df

train_df = add_velocity(train_df, 'uid_tx_count_24h')
val_df = add_velocity(val_df, 'uid_tx_count_24h')

# Mapeando os grupos de features
graph_cols = ['uid_pagerank', 'uid_degree', 'device_degree', 'uid_tx_count_24h']
fe_cols = [c for c in train_df.columns if c.endswith('n') and c[:-1] in ['D1', 'D2', 'D10', 'D15']] +           [c for c in train_df.columns if '_to_' in c] +           [c for c in train_df.columns if c.startswith('PCA_V_')]

print("✅ Features de Grafo criadas.")


---
## 5. Função de Treinamento e Avaliação Controlada (Ablação)


In [ ]:
def train_and_eval_pipeline(train_data, val_data, feature_set_name, config_type='all_in_one'):
    print(f"\n=======================================================")
    print(f" Treinando Pipeline: {feature_set_name}")
    print(f"=======================================================")
    
    # Definição dos conjuntos de variáveis
    base_cols_to_drop = ['isFraud', 'TransactionID', 'uid', 'timestamp', 'DeviceInfo_clean']
    
    # Roteador de Atributos com base na configuração da ablação
    if config_type == 'baseline':
        # Remove ambas
        cols_to_drop = base_cols_to_drop + graph_cols + fe_cols
    elif config_type == 'only_graphs':
        # Mantém grafos, remove feature engineering avançado
        cols_to_drop = base_cols_to_drop + fe_cols
    elif config_type == 'only_fe':
        # Mantém feature engineering avançado, remove grafos
        cols_to_drop = base_cols_to_drop + graph_cols
    else:  # all_in_one
        # Mantém tudo
        cols_to_drop = base_cols_to_drop

    features_a = [c for c in train_data.columns if c not in cols_to_drop and not c.startswith('id_') and c != 'DeviceType']
    features_b = [c for c in train_data.columns if c not in cols_to_drop]

    print(f"Variáveis Model A (Sem Dispositivo): {len(features_a)}")
    print(f"Variáveis Model B (Com Dispositivo): {len(features_b)}")

    scale_pos_a = (~train_data[~train_has_dev]['isFraud'].astype(bool)).sum() / max(train_data[~train_has_dev]['isFraud'].sum(), 1)
    scale_pos_b = (~train_data[train_has_dev]['isFraud'].astype(bool)).sum() / max(train_data[train_has_dev]['isFraud'].sum(), 1)

    t0 = time.time()
    
    # Treinamento do XGBoost para os Grupos A e B
    print("  -> Treinando XGBoost...")
    model_a_xgb = xgb.XGBClassifier(n_estimators=450, learning_rate=0.03, max_depth=9, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_a, tree_method='hist', random_state=42)
    model_a_xgb.fit(train_data.loc[~train_has_dev, features_a], train_data.loc[~train_has_dev, 'isFraud'], verbose=False)

    model_b_xgb = xgb.XGBClassifier(n_estimators=450, learning_rate=0.03, max_depth=9, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_b, tree_method='hist', random_state=42)
    model_b_xgb.fit(train_data.loc[train_has_dev, features_b], train_data.loc[train_has_dev, 'isFraud'], verbose=False)

    elapsed = time.time() - t0

    # Predição dos Modelos
    pred_a = model_a_xgb.predict_proba(val_data.loc[~val_has_dev, features_a])[:, 1]
    pred_b = model_b_xgb.predict_proba(val_data.loc[val_has_dev, features_b])[:, 1]

    y_score = pd.Series(index=val_data.index, dtype=float)
    y_score.loc[~val_has_dev] = pred_a
    y_score.loc[val_has_dev] = pred_b
    y_true = val_data['isFraud']

    # Pós-processamento de cliente se usar grafos
    if config_type in ['only_graphs', 'all_in_one']:
        val_data['raw_pred'] = y_score
        uid_pred_mean = val_data.groupby('uid')['raw_pred'].transform('mean')
        y_score = 0.7 * y_score + 0.3 * uid_pred_mean

    # Métricas
    roc = roc_auc_score(y_true, y_score)
    prec, rec, thresholds = precision_recall_curve(y_true, y_score)
    pr_auc = auc(rec, prec)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_thresh = thresholds[min(best_idx, len(thresholds)-1)]

    print(f"  --> Tempo: {elapsed:.1f}s | ROC-AUC: {roc:.4f} | PR-AUC: {pr_auc:.4f} | F1: {best_f1:.4f}")

    return {
        'name': feature_set_name,
        'roc_auc': roc,
        'pr_auc': pr_auc,
        'f1': best_f1,
        'threshold': best_thresh,
        'time': elapsed,
        'y_true': y_true,
        'y_score': y_score,
        'model_b_xgb': model_b_xgb,
        'features_b': features_b
    }


---
## 6. Execução do Experimento Comparativo Quadruplo


In [ ]:
# 1. Baseline XGBoost
res_baseline = train_and_eval_pipeline(train_df, val_df, "XGBoost Baseline", config_type='baseline')

# 2. XGBoost + ONLY Graph Features
res_only_graph = train_and_eval_pipeline(train_df, val_df, "XGBoost + Apenas Grafos", config_type='only_graphs')

# 3. XGBoost + ONLY Feature Engineering
res_only_fe = train_and_eval_pipeline(train_df, val_df, "XGBoost + Apenas Feature Eng.", config_type='only_fe')

# 4. All in One XGBoost
res_all_in_one = train_and_eval_pipeline(train_df, val_df, "XGBoost All-in-One", config_type='all_in_one')


---
## 7. Resultados Consolidados & Curvas Comparativas


In [ ]:
# Consolidação em DataFrame
summary_df = pd.DataFrame([
    {
        'Configuração': r['name'],
        'ROC-AUC': f"{r['roc_auc']:.4f}",
        'PR-AUC': f"{r['pr_auc']:.4f}",
        'F1-Score': f"{r['f1']:.4f}",
        'Tempo de Treino': f"{r['time']:.1f}s"
    }
    for r in [res_baseline, res_only_graph, res_only_fe, res_all_in_one]
])

print("=========================================================================================")
print("                   TABELA COMPARTIVA DO ESTUDO DE ABLAÇÃO QUADRUPLO                      ")
print("=========================================================================================")
display(summary_df)
print("=========================================================================================")


In [ ]:
# Visualização das Curvas Comparativas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

configs = [
    (res_baseline, PALETTE['baseline'], '--'),
    (res_only_graph, PALETTE['only_graph'], '-'),
    (res_only_fe, PALETTE['only_fe'], '-.'),
    (res_all_in_one, PALETTE['all_in_one'], '-')
]

# 1. Curva Precision-Recall
for r, color, style in configs:
    p, rec_val, _ = precision_recall_curve(r['y_true'], r['y_score'])
    ax1.plot(rec_val, p, color=color, linestyle=style, label=f"{r['name']} (PR-AUC = {r['pr_auc']:.4f})")

ax1.axhline(res_baseline['y_true'].mean(), color='gray', linestyle=':', label='Baseline Aleatório')
ax1.set_xlabel('Recall')
ax1.set_ylabel('Precision')
ax1.set_title('Curva Precision-Recall Comparativa')
ax1.legend()

# 2. Curva ROC
for r, color, style in configs:
    fpr, tpr, _ = roc_curve(r['y_true'], r['y_score'])
    ax2.plot(fpr, tpr, color=color, linestyle=style, label=f"{r['name']} (ROC-AUC = {r['roc_auc']:.4f})")

ax2.plot([0, 1], [0, 1], color='gray', linestyle=':')
ax2.set_xlabel('Taxa de Falsos Positivos (FPR)')
ax2.set_ylabel('Taxa de Verdadeiros Positivos (TPR)')
ax2.set_title('Curva ROC Comparativa')
ax2.legend()

plt.tight_layout()
plt.show()


---
## 8. Importância de Features no Modelo Completo (All-in-One)


In [ ]:
# Ganho médio por split no XGBoost All-in-One
importances = res_all_in_one['model_b_xgb'].get_booster().get_score(importance_type='gain')
imp_df = pd.DataFrame(list(importances.items()), columns=['Feature', 'Gain']).sort_values('Gain', ascending=False)

top25 = imp_df.head(25)

plt.figure(figsize=(10, 6))
colors = [PALETTE['only_graph'] if f in graph_cols else PALETTE['only_fe'] if f in fe_cols else PALETTE['baseline'] for f in top25['Feature']]
ax = sns.barplot(data=top25, x='Gain', y='Feature', palette=colors)
plt.title('Top 25 Variáveis Mais Importantes no XGBoost All-in-One')
plt.xlabel('Gain')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()
